In [0]:
%run ../../src/utils/config

In [0]:
%python
import os

config_path = os.path.abspath(
    os.path.join(os.getcwd(), "../../config/config.yml")
)

config = load_config(config_path)

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --------------------------------------------------
# Configuration
# --------------------------------------------------
CATALOG = config["target"]["catalog"]
BRONZE_SCHEMA = config["target"]["bronze_schema"]

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.pokemon_raw"

# --------------------------------------------------
# Read Bronze
# --------------------------------------------------
bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze table: {BRONZE_TABLE}")
print(f"Total Bronze rows: {bronze_df.count()}")

# --------------------------------------------------
# Keep successful API responses
# --------------------------------------------------
successful_df = (
    bronze_df
    .filter(F.col("status_code") == 200)
    .withColumn(
        "pokemon_id",
        F.get_json_object("raw_payload", "$.id").cast("int")
    )
)

# --------------------------------------------------
# Validate records before selecting the latest
# --------------------------------------------------
invalid_df = successful_df.filter(
    F.col("raw_payload").isNull()
    | F.col("pokemon_id").isNull()
    | F.col("_ingestion_timestamp").isNull()
)

invalid_count = invalid_df.count()

if invalid_count > 0:
    display(invalid_df.limit(20))
    raise ValueError(
        f"Found {invalid_count} successful Bronze records "
        "with a missing payload, Pokémon ID, or ingestion timestamp."
    )

# --------------------------------------------------
# Select the latest successful payload per Pokémon
# --------------------------------------------------
window_spec = (
    Window
    .partitionBy("pokemon_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc(),
        F.sha2(F.col("raw_payload"), 256).desc()
    )
)

latest_raw_df = (
    successful_df
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

# --------------------------------------------------
# Snapshot-level validation
# --------------------------------------------------
snapshot_count = latest_raw_df.count()
distinct_id_count = latest_raw_df.select("pokemon_id").distinct().count()

if snapshot_count == 0:
    raise ValueError("The latest Pokémon snapshot is empty.")

if snapshot_count != distinct_id_count:
    raise ValueError(
        "Duplicate Pokémon IDs found in the selected snapshot."
    )

print(f"Latest snapshot rows: {snapshot_count}")
print(f"Distinct Pokémon IDs: {distinct_id_count}")

display(
    latest_raw_df.select(
        "pokemon_id",
        "_ingestion_timestamp",
        "request_url",
        "status_code"
    ).orderBy("pokemon_id").limit(20)
)


In [0]:

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType,
    ArrayType
)

# --------------------------------------------------
# Reusable API reference schema
# --------------------------------------------------
named_api_object_schema = StructType([
    StructField("name", StringType(), True),
    StructField("url", StringType(), True)
])

# --------------------------------------------------
# Types
# --------------------------------------------------
type_entry_schema = StructType([
    StructField("slot", IntegerType(), True),
    StructField("type", named_api_object_schema, True)
])

# --------------------------------------------------
# Abilities
# --------------------------------------------------
ability_entry_schema = StructType([
    StructField("slot", IntegerType(), True),
    StructField("is_hidden", BooleanType(), True),
    StructField("ability", named_api_object_schema, True)
])

# --------------------------------------------------
# Stats
# --------------------------------------------------
stat_entry_schema = StructType([
    StructField("base_stat", IntegerType(), True),
    StructField("effort", IntegerType(), True),
    StructField("stat", named_api_object_schema, True)
])

# --------------------------------------------------
# Move version details
# --------------------------------------------------
move_learn_method_schema = StructType([
    StructField("name", StringType(), True),
    StructField("url", StringType(), True)
])

move_version_group_schema = StructType([
    StructField("name", StringType(), True),
    StructField("url", StringType(), True)
])

move_version_detail_schema = StructType([
    StructField("level_learned_at", IntegerType(), True),
    StructField("order", IntegerType(), True),
    StructField("move_learn_method", move_learn_method_schema, True),
    StructField("version_group", move_version_group_schema, True)
])

move_entry_schema = StructType([
    StructField("move", named_api_object_schema, True),
    StructField(
        "version_group_details",
        ArrayType(move_version_detail_schema),
        True
    )
])

# --------------------------------------------------
# Forms
# --------------------------------------------------
form_entry_schema = StructType([
    StructField("name", StringType(), True),
    StructField("url", StringType(), True)
])

# --------------------------------------------------
# Game indices
# --------------------------------------------------
game_index_schema = StructType([
    StructField("game_index", IntegerType(), True),
    StructField("version", named_api_object_schema, True)
])

# --------------------------------------------------
# Complete Pokémon response schema
# --------------------------------------------------
pokemon_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("base_experience", IntegerType(), True),
    StructField("height", IntegerType(), True),
    StructField("weight", IntegerType(), True),
    StructField("is_default", BooleanType(), True),
    StructField("order", IntegerType(), True),

    StructField("species", named_api_object_schema, True),

    StructField("types", ArrayType(type_entry_schema), True),
    StructField("abilities", ArrayType(ability_entry_schema), True),
    StructField("stats", ArrayType(stat_entry_schema), True),
    StructField("moves", ArrayType(move_entry_schema), True),
    StructField("forms", ArrayType(form_entry_schema), True),
    StructField("game_indices", ArrayType(game_index_schema), True)
])

print("Pokémon JSON schema defined successfully.")


In [0]:

# --------------------------------------------------
# Parse raw JSON into a structured DataFrame
# --------------------------------------------------
parsed_df = (
    latest_raw_df
    .withColumn(
        "pokemon",
        F.from_json(F.col("raw_payload"), pokemon_schema)
    )
)

# --------------------------------------------------
# Validate parsing
# --------------------------------------------------
parse_validation_df = parsed_df.select(
    "pokemon_id",
    F.col("pokemon.id").alias("parsed_pokemon_id"),
    F.col("pokemon.name").alias("pokemon_name"),
    F.col("pokemon.types").isNull().alias("types_is_null"),
    F.col("pokemon.abilities").isNull().alias("abilities_is_null"),
    F.col("pokemon.moves").isNull().alias("moves_is_null"),
    F.col("pokemon.stats").isNull().alias("stats_is_null"),
    F.col("pokemon.forms").isNull().alias("forms_is_null"),
    F.col("pokemon.game_indices").isNull().alias("game_indices_is_null"),
    "_ingestion_timestamp"
)

# A missing parsed ID indicates malformed JSON or an incompatible schema.
invalid_parse_df = parse_validation_df.filter(
    F.col("parsed_pokemon_id").isNull()
    | (F.col("pokemon_id") != F.col("parsed_pokemon_id"))
)

invalid_parse_count = invalid_parse_df.count()

if invalid_parse_count > 0:
    display(invalid_parse_df.limit(20))
    raise ValueError(
        f"{invalid_parse_count} payloads failed JSON parsing "
        "or have mismatched Pokémon IDs."
    )

print(f"Successfully parsed Pokémon: {parsed_df.count()}")

display(
    parse_validation_df
    .orderBy("pokemon_id")
    .limit(20)
)


In [0]:

# --------------------------------------------------
# Shared source columns
# --------------------------------------------------
source_df = parsed_df.select(
    "pokemon_id",
    F.col("pokemon"),
    F.col("_ingestion_timestamp").alias("source_ingestion_timestamp")
)

def add_silver_processed_at(df):
    """
    Add a processing timestamp to a Silver DataFrame.
    This value will be used when inserting or updating rows.
    """
    return df.withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )

# --------------------------------------------------
# 1. Pokémon: one row per Pokémon
# --------------------------------------------------
pokemon_silver_df = source_df.select(
    "pokemon_id",
    F.col("pokemon.name").alias("name"),
    F.col("pokemon.base_experience").alias("base_experience"),
    F.col("pokemon.height").alias("height"),
    F.col("pokemon.weight").alias("weight"),
    F.col("pokemon.is_default").alias("is_default"),
    F.col("pokemon.order").alias("pokemon_order"),
    F.col("pokemon.species.name").alias("species_name"),
    F.col("pokemon.species.url").alias("species_url"),
    "source_ingestion_timestamp"
)

pokemon_silver_df = add_silver_processed_at(pokemon_silver_df)

# --------------------------------------------------
# 2. Pokémon types: one row per Pokémon × type slot
# --------------------------------------------------
pokemon_types_df = (
    source_df
    .select(
        "pokemon_id",
        "source_ingestion_timestamp",
        F.explode("pokemon.types").alias("entry")
    )
    .select(
        "pokemon_id",
        F.col("entry.slot").alias("slot"),
        F.col("entry.type.name").alias("type_name"),
        F.col("entry.type.url").alias("type_url"),
        "source_ingestion_timestamp"
    )
)
pokemon_types_df = add_silver_processed_at(pokemon_types_df)

# --------------------------------------------------
# 3. Pokémon abilities: one row per Pokémon × ability slot
# --------------------------------------------------
pokemon_abilities_df = (
    source_df
    .select(
        "pokemon_id",
        "source_ingestion_timestamp",
        F.explode("pokemon.abilities").alias("entry")
    )
    .select(
        "pokemon_id",
        F.col("entry.slot").alias("slot"),
        F.col("entry.ability.name").alias("ability_name"),
        F.col("entry.ability.url").alias("ability_url"),
        F.col("entry.is_hidden").alias("is_hidden"),
        "source_ingestion_timestamp"
    )
)
pokemon_abilities_df = add_silver_processed_at(pokemon_abilities_df)

# --------------------------------------------------
# 4. Pokémon stats: one row per Pokémon × stat
# --------------------------------------------------
pokemon_stats_df = (
    source_df
    .select(
        "pokemon_id",
        "source_ingestion_timestamp",
        F.explode("pokemon.stats").alias("entry")
    )
    .select(
        "pokemon_id",
        F.col("entry.stat.name").alias("stat_name"),
        F.col("entry.stat.url").alias("stat_url"),
        F.col("entry.base_stat").alias("base_stat"),
        F.col("entry.effort").alias("effort"),
        "source_ingestion_timestamp"
    )
)
pokemon_stats_df = add_silver_processed_at(pokemon_stats_df)

# --------------------------------------------------
# 5. Pokémon moves: one row per Pokémon × move
# --------------------------------------------------
pokemon_moves_df = (
    source_df
    .select(
        "pokemon_id",
        "source_ingestion_timestamp",
        F.explode("pokemon.moves").alias("entry")
    )
    .select(
        "pokemon_id",
        F.col("entry.move.name").alias("move_name"),
        F.col("entry.move.url").alias("move_url"),
        "source_ingestion_timestamp"
    )
)
pokemon_moves_df = add_silver_processed_at(pokemon_moves_df)

# --------------------------------------------------
# 6. Move details: one row per Pokémon × move ×
#    version group × learning method × detail key
# --------------------------------------------------
pokemon_move_details_df = (
    source_df
    .select(
        "pokemon_id",
        "source_ingestion_timestamp",
        F.explode("pokemon.moves").alias("move_entry")
    )
    .select(
        "pokemon_id",
        "source_ingestion_timestamp",
        F.col("move_entry.move.name").alias("move_name"),
        F.explode("move_entry.version_group_details").alias("detail")
    )
    .select(
        "pokemon_id",
        "move_name",
        F.col("detail.version_group.name").alias("version_group_name"),
        F.col("detail.version_group.url").alias("version_group_url"),
        F.col("detail.move_learn_method.name").alias("learn_method_name"),
        F.col("detail.move_learn_method.url").alias("learn_method_url"),
        F.col("detail.level_learned_at").alias("level_learned_at"),
        F.col("detail.order").alias("learn_order"),
        "source_ingestion_timestamp"
    )
)
pokemon_move_details_df = add_silver_processed_at(
    pokemon_move_details_df
)

# --------------------------------------------------
# 7. Pokémon forms: one row per Pokémon × form
# --------------------------------------------------
pokemon_forms_df = (
    source_df
    .select(
        "pokemon_id",
        "source_ingestion_timestamp",
        F.explode("pokemon.forms").alias("entry")
    )
    .select(
        "pokemon_id",
        F.col("entry.name").alias("form_name"),
        F.col("entry.url").alias("form_url"),
        "source_ingestion_timestamp"
    )
)
pokemon_forms_df = add_silver_processed_at(pokemon_forms_df)

# --------------------------------------------------
# 8. Game indices: one row per Pokémon × version
# --------------------------------------------------
pokemon_game_indices_df = (
    source_df
    .select(
        "pokemon_id",
        "source_ingestion_timestamp",
        F.explode("pokemon.game_indices").alias("entry")
    )
    .select(
        "pokemon_id",
        F.col("entry.game_index").alias("game_index"),
        F.col("entry.version.name").alias("version_name"),
        F.col("entry.version.url").alias("version_url"),
        "source_ingestion_timestamp"
    )
)
pokemon_game_indices_df = add_silver_processed_at(
    pokemon_game_indices_df
)

print("Created all eight Silver DataFrames.")


In [0]:

silver_candidates = {
    "pokemon": (
        pokemon_silver_df,
        ["pokemon_id"]
    ),
    "pokemon_types": (
        pokemon_types_df,
        ["pokemon_id", "slot"]
    ),
    "pokemon_abilities": (
        pokemon_abilities_df,
        ["pokemon_id", "slot"]
    ),
    "pokemon_stats": (
        pokemon_stats_df,
        ["pokemon_id", "stat_name"]
    ),
    "pokemon_moves": (
        pokemon_moves_df,
        ["pokemon_id", "move_name"]
    ),
    "pokemon_move_details": (
        pokemon_move_details_df,
        [
            "pokemon_id",
            "move_name",
            "version_group_name",
            "learn_method_name",
            "level_learned_at",
            "learn_order"
        ]
    ),
    "pokemon_forms": (
        pokemon_forms_df,
        ["pokemon_id", "form_name"]
    ),
    "pokemon_game_indices": (
        pokemon_game_indices_df,
        ["pokemon_id", "version_name"]
    )
}

for table_name, (df, key_columns) in silver_candidates.items():
    row_count = df.count()

    duplicate_key_count = (
        df.groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(
        f"{table_name}: rows={row_count}, "
        f"duplicate_keys={duplicate_key_count}"
    )

    if duplicate_key_count > 0:
        print(f"Duplicate keys found in {table_name}:")
        display(
            df.groupBy(*key_columns)
            .count()
            .filter(F.col("count") > 1)
            .limit(20)
        )


In [0]:

# --------------------------------------------------
# Required-field validation rules
# --------------------------------------------------
required_fields = {
    "pokemon": [
        "pokemon_id",
        "name",
        "source_ingestion_timestamp"
    ],
    "pokemon_types": [
        "pokemon_id",
        "slot",
        "type_name",
        "type_url",
        "source_ingestion_timestamp"
    ],
    "pokemon_abilities": [
        "pokemon_id",
        "slot",
        "ability_name",
        "ability_url",
        "is_hidden",
        "source_ingestion_timestamp"
    ],
    "pokemon_stats": [
        "pokemon_id",
        "stat_name",
        "stat_url",
        "base_stat",
        "effort",
        "source_ingestion_timestamp"
    ],
    "pokemon_moves": [
        "pokemon_id",
        "move_name",
        "move_url",
        "source_ingestion_timestamp"
    ],
    "pokemon_move_details": [
        "pokemon_id",
        "move_name",
        "version_group_name",
        "version_group_url",
        "learn_method_name",
        "learn_method_url",
        "level_learned_at",
        "source_ingestion_timestamp"
    ],
    "pokemon_forms": [
        "pokemon_id",
        "form_name",
        "form_url",
        "source_ingestion_timestamp"
    ],
    "pokemon_game_indices": [
        "pokemon_id",
        "game_index",
        "version_name",
        "version_url",
        "source_ingestion_timestamp"
    ]
}

# --------------------------------------------------
# Run required-field checks
# --------------------------------------------------
validation_failures = []

for table_name, (df, key_columns) in silver_candidates.items():
    for column_name in required_fields[table_name]:
        null_count = df.filter(
            F.col(column_name).isNull()
        ).count()

        if null_count > 0:
            validation_failures.append({
                "table": table_name,
                "column": column_name,
                "null_count": null_count
            })

if validation_failures:
    print("Required-field validation failed:")
    display(spark.createDataFrame(validation_failures))
    raise ValueError("Silver required-field validation failed.")

print("All required-field checks passed.")

# --------------------------------------------------
# Referential integrity: child rows must reference
# Pokémon present in the current parent snapshot
# --------------------------------------------------
parent_ids = pokemon_silver_df.select("pokemon_id").distinct()

child_tables = [
    "pokemon_types",
    "pokemon_abilities",
    "pokemon_stats",
    "pokemon_moves",
    "pokemon_move_details",
    "pokemon_forms",
    "pokemon_game_indices"
]

for table_name in child_tables:
    child_df = silver_candidates[table_name][0]

    orphan_df = (
        child_df
        .select("pokemon_id")
        .distinct()
        .join(parent_ids, on="pokemon_id", how="left_anti")
    )

    orphan_count = orphan_df.count()

    print(f"{table_name}: orphan Pokémon IDs = {orphan_count}")

    if orphan_count > 0:
        display(orphan_df.limit(20))
        raise ValueError(
            f"Referential integrity validation failed for {table_name}."
        )

print("All Silver candidate DataFrames passed validation.")


In [0]:

# --------------------------------------------------
# Snapshot consistency gate
# No hardcoded expected Pokémon count
# --------------------------------------------------

snapshot_row_count = latest_raw_df.count()
snapshot_ids_df = latest_raw_df.select("pokemon_id").distinct()
snapshot_id_count = snapshot_ids_df.count()

# The selected snapshot must be non-empty and contain unique IDs.
if snapshot_row_count == 0:
    raise ValueError("Refusing to process an empty Pokémon snapshot.")

if snapshot_row_count != snapshot_id_count:
    raise ValueError(
        "Snapshot contains duplicate Pokémon IDs."
    )

# Confirm the parsed parent DataFrame matches the Bronze snapshot.
parsed_ids_df = pokemon_silver_df.select("pokemon_id").distinct()

missing_from_parsed = snapshot_ids_df.join(
    parsed_ids_df,
    on="pokemon_id",
    how="left_anti"
).count()

extra_in_parsed = parsed_ids_df.join(
    snapshot_ids_df,
    on="pokemon_id",
    how="left_anti"
).count()

if missing_from_parsed > 0 or extra_in_parsed > 0:
    raise ValueError(
        "Parsed Pokémon IDs do not match the selected Bronze snapshot."
    )

# Confirm child tables only reference Pokémon in this snapshot.
for table_name, (df, key_columns) in silver_candidates.items():
    unexpected_ids = (
        df.select("pokemon_id").distinct()
        .join(snapshot_ids_df, on="pokemon_id", how="left_anti")
        .count()
    )

    if unexpected_ids > 0:
        raise ValueError(
            f"{table_name} contains Pokémon IDs outside the snapshot."
        )

print("Snapshot consistency gate passed.")
print(f"Snapshot rows: {snapshot_row_count}")
print(f"Distinct Pokémon IDs: {snapshot_id_count}")
print(f"Validated Silver DataFrames: {len(silver_candidates)}")


In [0]:

# DataFrames are produced by the transformation logic.
silver_dataframes = {
    "pokemon": pokemon_silver_df,
    "types": pokemon_types_df,
    "abilities": pokemon_abilities_df,
    "stats": pokemon_stats_df,
    "moves": pokemon_moves_df,
    "move_details": pokemon_move_details_df,
    "forms": pokemon_forms_df,
    "game_indices": pokemon_game_indices_df
}

# Load keys and targets from YAML configuration.
merge_config = {}

for dataset_name, settings in config["silver"]["datasets"].items():
    source_entity = settings["source_entity"]

    merge_config[dataset_name] = {
        "df": silver_dataframes[source_entity],
        "target_table": settings["target_table"],
        "load_strategy": settings["load_strategy"],
        "keys": settings["keys"]
    }

print(f"Configured {len(merge_config)} Silver targets.")


In [0]:

def silver_table_name(table_name: str) -> str:
    return (
        f"{config['target']['catalog']}."
        f"{config['target']['silver_schema']}."
        f"{table_name}"
    )

for dataset_name, settings in merge_config.items():
    print(
        f"{dataset_name} -> "
        f"{silver_table_name(settings['target_table'])}; "
        f"keys={settings['keys']}"
    )


In [0]:

# --------------------------------------------------
# Pre-merge target validation
# --------------------------------------------------

for dataset_name, settings in merge_config.items():
    target_table = silver_table_name(settings["target_table"])
    source_df = settings["df"]

    if not spark.catalog.tableExists(target_table):
        raise ValueError(
            f"Target table does not exist: {target_table}"
        )

    source_columns = set(source_df.columns)
    target_columns = set(spark.table(target_table).columns)

    if source_columns != target_columns:
        raise ValueError(
            f"Schema mismatch for {target_table}. "
            f"Source-only columns: {sorted(source_columns - target_columns)}; "
            f"Target-only columns: {sorted(target_columns - source_columns)}"
        )

    missing_keys = set(settings["keys"]) - source_columns

    if missing_keys:
        raise ValueError(
            f"Merge keys missing from {dataset_name}: {sorted(missing_keys)}"
        )

    print(f"Ready: {target_table}")

print("All eight Silver targets passed pre-merge validation.")


In [0]:

from delta.tables import DeltaTable

def merge_silver_upsert(source_df, target_table, key_columns):
    """
    Insert new records and update changed records.
    Does not delete records missing from the source.
    """

    # Fail if the source is empty.
    if source_df.limit(1).count() == 0:
        raise ValueError(
            f"Source is empty. Refusing to merge into {target_table}."
        )

    # Verify source and target schemas match.
    source_columns = set(source_df.columns)
    target_columns = set(spark.table(target_table).columns)

    if source_columns != target_columns:
        raise ValueError(
            f"Schema mismatch for {target_table}. "
            f"Source-only: {sorted(source_columns - target_columns)}; "
            f"Target-only: {sorted(target_columns - source_columns)}"
        )

    # Verify the merge keys exist.
    if not set(key_columns).issubset(source_columns):
        raise ValueError(
            f"Merge keys missing from source for {target_table}."
        )

    # Null-safe key equality handles nullable learn_order.
    merge_condition = " AND ".join(
        f"t.`{key}` <=> s.`{key}`"
        for key in key_columns
    )

    # Exclude keys and processing timestamp from change detection.
    comparison_columns = [
        column for column in source_df.columns
        if column not in key_columns
        and column != "silver_processed_at"
    ]

    change_condition = " OR ".join(
        f"NOT (t.`{column}` <=> s.`{column}`)"
        for column in comparison_columns
    ) or "false"

    # Update business/source fields only when data changed.
    update_values = {
        column: f"s.`{column}`"
        for column in source_df.columns
        if column != "silver_processed_at"
    }
    update_values["silver_processed_at"] = "current_timestamp()"

    insert_values = {
        column: f"s.`{column}`"
        for column in source_df.columns
    }

    target = DeltaTable.forName(spark, target_table)

    (
        target.alias("t")
        .merge(source_df.alias("s"), merge_condition)
        .whenMatchedUpdate(
            condition=change_condition,
            set=update_values
        )
        .whenNotMatchedInsert(values=insert_values)
        .execute()
    )

    print(f"Upsert completed: {target_table}")


In [0]:

for dataset_name, settings in merge_config.items():
    target_table = silver_table_name(settings["target_table"])

    if settings["load_strategy"] != "snapshot_merge":
        raise ValueError(
            f"Unsupported load strategy for {dataset_name}: "
            f"{settings['load_strategy']}"
        )

    merge_silver_upsert(
        source_df=settings["df"],
        target_table=target_table,
        key_columns=settings["keys"]
    )

print("All configured Silver upserts completed.")


In [0]:

for dataset_name, settings in merge_config.items():
    target_table = silver_table_name(settings["target_table"])
    key_columns = settings["keys"]

    target_df = spark.table(target_table)

    row_count = target_df.count()

    duplicate_key_count = (
        target_df.groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(
        f"{target_table}: rows={row_count}, "
        f"duplicate_keys={duplicate_key_count}"
    )

    if duplicate_key_count > 0:
        raise ValueError(
            f"Duplicate merge keys found in {target_table}."
        )

print("Silver post-merge validation completed.")
